In [15]:
from dotenv import load_dotenv
from pprint import pprint
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain.messages import HumanMessage

load_dotenv()

True

In [16]:
client = MultiServerMCPClient(
    {
        "math": {
            "command": "python",
            "args": ["resources/2.1_math_mcp_server.py"],
            "transport": "stdio"
        },
        "weather": {
            # Make sure you start your weather server on port 8001
            "url": "http://localhost:8001/mcp",
            "transport": "streamable_http",
        }
    }
)
tools = await client.get_tools()
resources = await client.get_resources()

print("tools:")
pprint(tools)
print("\nresources:")
pprint(resources)

tools:
[StructuredTool(name='add', description='Add two numbers', args_schema={'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'addArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x118c9e520>),
 StructuredTool(name='multiply', description='Multiply two numbers', args_schema={'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiplyArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x11a046340>),
 StructuredTool(name='get_weather', description='Get weather for location.', args_schema={'additionalProperties': False, 'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}, metadata={'_meta': {'fastmcp

In [17]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt="""
        You are an assistant that can only answer questions about math (addition & multiplication) and weather of a location.
        You may try multiple tool and resource calls to answer the user's question.
        You may also ask clarifying questions to the user to better understand their question.
        If you don't know the answer, you can say "Sorry, I don't know". Do not try to make up an answer.
    """
)

In [18]:
response = await agent.ainvoke(input={"messages": [HumanMessage(content="what's redis usage?")]})
pprint(response)

{'messages': [HumanMessage(content="what's redis usage?", additional_kwargs={}, response_metadata={}, id='b23761d3-39ce-43f1-a1ad-ec92e2c5cf91'),
              AIMessage(content="Sorry, I don't know.\n\nI can help with weather for a location or with math problems involving addition or multiplication. If you want weather, tell me the location. If you have a math problem, give me the numbers to add or multiply.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 506, 'prompt_tokens': 256, 'total_tokens': 762, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWAEhZSJIPJZJ92ypucxEwvgYclkH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da31f-0

In [19]:
math_response = await agent.ainvoke(input={"messages": [HumanMessage(content="what's (3 + 5) x 12?")]})
weather_response = await agent.ainvoke(input={"messages": [HumanMessage(content="what is the weather in NYC?")]})

pprint(math_response)
pprint(weather_response)

{'messages': [HumanMessage(content="what's (3 + 5) x 12?", additional_kwargs={}, response_metadata={}, id='cc8408df-78d5-42e1-b676-1ff8bc905f4b'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 346, 'prompt_tokens': 263, 'total_tokens': 609, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWAEoeIgxREEGybI8kYG6JvDagcZA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da31f-296c-7291-b501-e234e3168b00-0', tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': 'call_PaYZEMEJoKuRLrV2az6fH6qL', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 263, 'output_tokens': 34